In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# -----------------------------
# 1. Get vars (widgets + defaults)
# -----------------------------

def get_widget_or_default(name: str, default: str) -> str:
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

DEFAULT_CATALOG = "eligibility_operation"
DEFAULT_SCHEMA = "default"
DEFAULT_STAGING_TABLE = "member_eligibility_staging"
DEFAULT_REJECTED_TABLE = "member_eligibility_rejected"
TARGET_TABLE = "member_eligibility"

catalog = get_widget_or_default("catalog", DEFAULT_CATALOG)
schema = get_widget_or_default("schema", DEFAULT_SCHEMA)
staging_table_name = get_widget_or_default("staging_table", DEFAULT_STAGING_TABLE)
rejected_table_name = get_widget_or_default("rejected_table", DEFAULT_REJECTED_TABLE)

db = f"{catalog}.{schema}"
staging_table = f"{db}.{staging_table_name}"
rejected_table = f"{db}.{rejected_table_name}"
target_table = f"{db}.{TARGET_TABLE}"

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

# -----------------------------
# 2. Read from staging (PENDING only for idempotency)
# -----------------------------

staging_df = (
    spark.table(staging_table)
)

print(f"Records to validate: {staging_df.count()}")
# display(staging_df)

# -----------------------------
# 3. Apply validations
# -----------------------------

external_id_invalid = col("external_id").isNull()

dob_parsed = when(
    col("dob").rlike(r"^\d{4}-\d{2}-\d{2}$"), to_date(col("dob"), "yyyy-MM-dd")
).otherwise(lit(None))
dob_invalid = dob_parsed.isNull() & col("dob").isNotNull()

# FIXED: Proper email null handling
email_invalid = (
    col("email").isNull() |
    (
        col("email").isNotNull() &
        ~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
    )
)

df_with_flags = (
    staging_df
    .withColumn(
        "external_id_error",
        when(external_id_invalid, lit("external_id is null"))
    )
    .withColumn(
        "dob_error",
        when(dob_invalid, lit("dob not in yyyy-MM-dd"))
    )
    .withColumn(
        "email_error",
        when(email_invalid, lit("invalid email format"))
    )
)


df_with_errors = (
    df_with_flags
    .withColumn(
        "field_errors",
        array_compact(  # removes all nulls automatically
            array(
                when(col("external_id_error").isNotNull(), lit("external_id")),
                when(col("dob_error").isNotNull(), lit("dob")),
                when(col("email_error").isNotNull(), lit("email"))
            )
        )
    )
    .withColumn(
        "field_errors",
        when(size(col("field_errors")) > 0, array_join(col("field_errors"), "; "))
    )
)

df_with_errors = (
    df_with_errors
    .withColumn(
        "error_codes",
        array_compact(
            array(
                when(external_id_invalid, lit("EXT_NULL")),
                when(dob_invalid, lit("DOB_FMT")),
                when(email_invalid, lit("EMAIL_FMT"))
            )
        )
    )
    .withColumn(
        "error_codes",
        when(size(col("error_codes")) > 0, array_join(col("error_codes"), ","))
    )
)

df_with_errors = df_with_errors.withColumn(
    "is_invalid",
    col("field_errors").isNotNull()
)

display(df_with_errors)

invalid_df = df_with_errors.filter(col("is_invalid"))
valid_df = df_with_errors.filter(~col("is_invalid"))

# Display counts
invalid_count = invalid_df.count()
valid_count = valid_df.count()
print(f"Invalid records: {invalid_count}")
print(f"Valid records: {valid_count}")

# display(invalid_df)


Records to validate: 10


staging_id,partner_id,partner_code,external_id,first_name,last_name,dob,email,phone,validation_status,validation_errors,created_date,external_id_error,dob_error,email_error,field_errors,error_codes,is_invalid
1,1,acme,1234567890A,John,Doe,03/15/1955,john.doe@email.com,5551234567,PENDING,Unknown,null,null,dob not in yyyy-MM-dd,null,dob,DOB_FMT,true
2,1,acme,9876543210B,Jane,Smith,07/22/1948,jane.smith@email.com,5559876543,PENDING,Unknown,null,null,dob not in yyyy-MM-dd,null,dob,DOB_FMT,true
3,1,acme,null,Alice,Johnson,08/10/1965,alice.j@test.com,5552223333,PENDING,Unknown,null,external_id is null,dob not in yyyy-MM-dd,null,external_id; dob,"EXT_NULL,DOB_FMT",true
4,1,acme,4567890123C,Bob123,Williams,InvalidDate,bob.williams@example.com,555-444-5555,PENDING,Unknown,null,null,dob not in yyyy-MM-dd,null,dob,DOB_FMT,true
5,1,acme,5678901234D,Charlie,Brown,1972-03-25,CHARLIE.BROWN@TEST.COM,(555) 333-4444,PENDING,Unknown,null,null,null,null,null,null,false
6,1,acme,6789012345E,Diana,O'Brien,11/30/1980,diana.obrien@mail.com,5556667777,PENDING,Unknown,null,null,dob not in yyyy-MM-dd,null,dob,DOB_FMT,true
7,1,acme,NULL,Edward,Davis,2030-05-15,edward.davis@email.com,555,PENDING,Unknown,null,null,null,null,null,null,false
8,1,acme,8901234567F,FionaFionaFionaFionaFionaFionaFiona,Miller,06/18/1968,fiona.miller@test.com,5558889999,PENDING,Unknown,null,null,dob not in yyyy-MM-dd,null,dob,DOB_FMT,true
9,1,acme,9012345678G,George,Taylor,09/25/1975,george.taylor@example.com,5551112222,PENDING,Unknown,null,null,dob not in yyyy-MM-dd,null,dob,DOB_FMT,true
10,1,acme,0123456789H,Hannah,Moore,12/05/1958,hannah@,5553334444,PENDING,Unknown,null,null,dob not in yyyy-MM-dd,invalid email format,dob; email,"DOB_FMT,EMAIL_FMT",true


Invalid records: 8
Valid records: 2


In [0]:

# -----------------------------
# 3b. Simple metrics (optional but useful)
# -----------------------------

total_count = staging_df.count()
valid_count = valid_df.count()
invalid_count = invalid_df.count()
print(f"Processed: {total_count}, Valid: {valid_count}, Invalid: {invalid_count}")

# -----------------------------
# 4. Write invalid rows to member_eligibility_rejected
# -----------------------------
# FIX: use UUID instead of hash for reject_id

from pyspark.sql.functions import expr

rejected_out_df = (
    invalid_df.withColumn("reject_id", monotonically_increasing_id())\
    .select(
        "reject_id",
        "partner_id",
        "partner_code",  # Add this column
        col("external_id").alias("external_id_sample"),
        "error_codes",
        "field_errors",
        current_timestamp().alias("rejected_date")
    )
)

(
    rejected_out_df
    .write
    .mode("append")
    .saveAsTable(rejected_table)
)

# -----------------------------
# 5. Update staging table rows (status)
# -----------------------------

invalid_updates = invalid_df.select(
    "staging_id",
    lit("FINISHED").alias("validation_status"),
    col("field_errors").alias("validation_errors")
)

valid_updates = valid_df.select(
    "staging_id",
    lit("FINISHED").alias("validation_status"),
    lit(None).cast("string").alias("validation_errors")
)

updates_df = invalid_updates.unionByName(valid_updates)

staging_delta = DeltaTable.forName(spark, staging_table)

(
    staging_delta.alias("t")
    .merge(
        updates_df.alias("u"),
        "t.staging_id = u.staging_id"
    )
    .whenMatchedUpdate(
        set={
            "validation_status": "u.validation_status",
            "validation_errors": "u.validation_errors"
        }
    )
    .execute()
)



Processed: 10, Valid: 2, Invalid: 8
